# Lesson 6.5 - LLMOps: Operationalizing LLMs, RAG & Agents (toy pipeline)

This notebook builds a compact RAG-like workflow with explicit observability logs for prompts, retrieval context, and generated answers.

## Objectives

- Index a small corpus and retrieve relevant passages.
- Generate grounded answers from retrieved context.
- Log LLMOps-relevant telemetry (prompt, retrieved chunks, latency-like fields).
- Demonstrate simple tool/agent routing behavior.

## Setup

In [ ]:
from __future__ import annotations

import json
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ARTIFACT_DIR = Path("artifacts/lesson6_5")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## Section A - Build a Tiny Knowledge Corpus

In [ ]:
docs = {
    "mlops_guide": "MLOps lifecycle includes data ingestion, training, deployment, monitoring, and feedback loops.",
    "monitoring_guide": "Data drift and prediction drift monitoring are essential for maintaining model quality in production.",
    "rag_guide": "RAG combines retrieval with generation. High-quality retrieval improves grounded answers and reduces hallucinations.",
    "deployment_guide": "Blue-green and canary deployments reduce risk when releasing new model versions.",
    "llmops_guide": "LLMOps adds prompt versioning, retrieval observability, cost controls, and safety policies to MLOps.",
}

chunks = []
for doc_id, text in docs.items():
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    for idx, sent in enumerate(sentences):
        chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_{idx}", "text": sent})

chunks_df = pd.DataFrame(chunks)
chunks_df

## Section B - Index + Retrieval

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
chunk_matrix = vectorizer.fit_transform(chunks_df["text"])


def retrieve(query: str, top_k: int = 3) -> pd.DataFrame:
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, chunk_matrix).ravel()
    idx = np.argsort(-scores)[:top_k]
    out = chunks_df.iloc[idx].copy()
    out["score"] = scores[idx]
    return out[["doc_id", "chunk_id", "score", "text"]].reset_index(drop=True)


retrieve("How should we monitor production ML models?")

## Section C - Toy Generation + LLMOps Logging

In [ ]:
logs = []


def generate_grounded_answer(query: str, top_k: int = 3) -> dict:
    t0 = time.time()
    retrieved = retrieve(query, top_k=top_k)

    if retrieved.empty:
        answer = "I do not have enough retrieved evidence to answer confidently."
    else:
        evidence = " ".join(retrieved["text"].tolist())
        answer = f"Based on retrieved evidence: {evidence}."

    latency_ms = int((time.time() - t0) * 1000)

    log_record = {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "query": query,
        "top_k": top_k,
        "retrieved_chunk_ids": retrieved["chunk_id"].tolist(),
        "retrieved_scores": [float(x) for x in retrieved["score"].tolist()],
        "answer": answer,
        "token_proxy_input_chars": len(query),
        "token_proxy_output_chars": len(answer),
        "latency_ms": latency_ms,
    }
    logs.append(log_record)

    return {"answer": answer, "retrieved": retrieved, "log": log_record}


result = generate_grounded_answer("How does LLMOps differ from MLOps?")
result["answer"]

In [ ]:
log_df = pd.DataFrame(logs)
log_df

## Section D - Simple Tool/Agent Routing

In [ ]:
def calculator_tool(expr: str) -> str:
    safe_chars = set("0123456789+-*/(). ")
    if not set(expr) <= safe_chars:
        return "calculator_error: invalid characters"
    try:
        return f"calculator_result: {eval(expr, {'__builtins__': {}}, {})}"
    except Exception as exc:
        return f"calculator_error: {exc}"


def agent(query: str) -> str:
    q = query.lower().strip()
    if any(op in q for op in ["+", "-", "*", "/"]) and any(ch.isdigit() for ch in q):
        expr = "".join(ch for ch in q if ch in "0123456789+-*/(). ")
        return calculator_tool(expr)

    return generate_grounded_answer(query)["answer"]


for q in [
    "What is RAG used for in production?",
    "What is 12 * (5 + 3)?",
    "Why do teams need model monitoring?",
]:
    print("Q:", q)
    print("A:", agent(q))
    print("-" * 80)

## Section E - Persist Observability Logs

In [ ]:
log_path = ARTIFACT_DIR / "llmops_query_logs.jsonl"
with log_path.open("w", encoding="utf-8") as f:
    for row in logs:
        f.write(json.dumps(row) + "
")

log_path

## Connect to Theory

- Retrieval + generation mirrors the core RAG pipeline.
- The `logs` structure mirrors LLMOps observability needs: prompts, retrieved evidence, outputs, and cost/latency proxies.
- The simple `agent` function demonstrates tool routing and highlights why tool safety controls are necessary.

## Business Case Studies & Exceptions

### Case 1: Enterprise policy assistant

- **Pattern**: enforce citation-required answers from retrieved policy chunks.
- **Benefit**: improved groundedness and auditability.
- **Exception**: stale index snapshots can produce outdated but fluent responses.

### Case 2: Cost spike in LLM application

- **Pattern**: monitor token usage and set budget guards + prompt compression.
- **Benefit**: predictable spend.
- **Exception**: aggressive truncation can hurt answer quality.

### Case 3: Agent tool misuse

- **Pattern**: strict tool schemas and allowlists with audit logs.
- **Benefit**: lower operational and security risk.

## Interview Questions & Answers

1. **What is LLMOps?**  
   The operational discipline for deploying, monitoring, and governing LLM applications in production.

2. **How does RAG help LLM systems?**  
   It injects retrieved external context to improve groundedness and freshness.

3. **What should be logged in a production RAG system?**  
   Prompt, retrieved chunks/scores, model output, latency, and cost-related telemetry.

4. **Why is retrieval quality critical?**  
   Weak retrieval degrades factual quality even when the base model is strong.

5. **How do you detect silent RAG degradation?**  
   Track retrieval metrics and groundedness over time, not only response fluency.

6. **What is a token budget?**  
   A limit on prompt/response tokens to control latency and cost.

7. **Why version prompts?**  
   Prompt changes can materially alter behavior and must be traceable for rollback.

8. **How do agents increase operational risk?**  
   Multi-step tool execution can amplify errors without strict permissions and validation.

9. **RAG vs fine-tuning in production?**  
   RAG updates knowledge via index refresh; fine-tuning updates model behavior and is slower to refresh facts.

10. **What is groundedness?**  
   The extent to which answer claims are supported by retrieved evidence.

11. **When should a system abstain instead of answering?**  
   When retrieval confidence is low or evidence is missing/contradictory.

12. **What is the first LLMOps improvement for a prototype app?**  
   Add observability logs and a repeatable evaluation set before scaling complexity.